In [ ]:
!pip install torch torchvision transformers datasets -q
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
print(torch.__version__, "GPU:", torch.cuda.is_available())

In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.Wxh = nn.Linear(input_size, hidden_size)
        self.Whh = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x):  # x: (batch, seq_len, input_size)
        batch, seq_len, _ = x.shape
        h = torch.zeros(batch, self.hidden_size)
        hidden_states = []
        for t in range(seq_len):
            h = torch.tanh(self.Wxh(x[:, t, :]) + self.Whh(h))
            h.retain_grad()          # keep gradient at each timestep
            hidden_states.append(h)
        return hidden_states

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm_cell = nn.LSTMCell(input_size, hidden_size)
        self.hidden_size = hidden_size

    def forward(self, x):
        batch, seq_len, _ = x.shape
        h = torch.zeros(batch, self.hidden_size)
        c = torch.zeros(batch, self.hidden_size)
        hidden_states = []
        for t in range(seq_len):
            h, c = self.lstm_cell(x[:, t, :], (h, c))
            h.retain_grad()
            hidden_states.append(h)
        return hidden_states

In [ ]:
def get_gradient_norms(model, x, seq_len):
    hidden_states = model(x)
    loss = hidden_states[-1].sum()   # backprop from the LAST timestep
    loss.backward()
    norms = [h.grad.norm().item() if h.grad is not None else 0 for h in hidden_states]
    return norms

seq_len = 50
x = torch.randn(4, seq_len, 20)

rnn = VanillaRNN(20, 32)
rnn_norms = get_gradient_norms(rnn, x, seq_len)

x2 = x.clone().detach().requires_grad_()
lstm = SimpleLSTM(20, 32)
lstm_norms = get_gradient_norms(lstm, x2, seq_len)

plt.plot(rnn_norms, label="Vanilla RNN")
plt.plot(lstm_norms, label="LSTM")
plt.xlabel("Timestep"); plt.ylabel("Gradient norm at that timestep")
plt.yscale("log")
plt.legend(); plt.title("Vanishing Gradient: RNN vs LSTM")
plt.show()